# Lesson 8 exercise: evaluating an extractor

**Goal:** build an honest before/after table on held-out rows, then write your own stress tests, find where the fine-tuned model breaks, and name each failure.

**How to use this notebook**
1. In Colab: *Runtime → Change runtime type → T4 GPU*.
2. Run the cells top to bottom. The whole notebook takes about 5 minutes.
3. Wherever you see `...` after a `# TODO`, replace it with your code.
4. Run the **check** cell under each TODO. ✅ means you've got it; ❌ gives you a hint.

Stuck? Look up the error in the course's troubleshooting sheet (`reference/troubleshooting.html`). Still stuck? Ask on the Hugging Face Forums with your code and the full traceback.

## 0 · Setup, task data and metric
The same three cells as Lessons 5 and 6: versions, the extraction data, and `score()`.

In [ ]:
%pip install -q -U "transformers>=5.19,<6" "trl>=1.15,<2" "peft>=0.21,<1" "datasets>=5,<6" "accelerate>=1.15,<2"
import torch, transformers
device = "cuda" if torch.cuda.is_available() else "cpu"
print("transformers", transformers.__version__, "| torch", torch.__version__, "| device:", device)

def check(passed, ok_msg, hint):
    print(("✅ " + ok_msg) if passed else ("❌ " + hint))

In [ ]:
import json, random

PROMPT = "Extract the person's name, age and city as JSON."
NAMES = ["Sara", "Omar", "Lina", "Youssef", "Mona", "Karim", "Nour", "Emma", "Liam", "Sofia",
         "Mateo", "Aiko", "Kenji", "Priya", "Arjun", "Chloe", "Lucas", "Fatima", "Diego", "Elena",
         "Leo", "Amara", "Kofi", "Hana", "Ines"]          # last 5 are test-only
CITIES = ["Cairo", "Alexandria", "Paris", "Lagos", "Tokyo", "Lima", "Berlin", "Toronto",
          "Mumbai", "Seoul", "Nairobi", "Madrid", "Dubai", "Oslo", "Austin", "Lisbon",
          "Hanoi", "Accra", "Quito", "Perth"]                # last 4 are test-only
TEMPLATES = [
    "Hi, I'm {name} from {city}. I'm {age}.",
    "{name} ({age}) lives in {city}.",
    "My name is {name}, I am {age} years old and I live in {city}.",
    "New signup: {name}, age {age}, based in {city}.",
    "Hello from {city}! {name} here, just turned {age}.",
    "{city}-based {name}, {age}, is joining the team next week.",
    "Contact: {name} | {age} | {city}",
    "Applicant {name} is {age} and currently resides in {city}.",
    "Meet {name}: {age} years young and loving life in {city}.",   # test-only
    "I moved to {city} last year. Name's {name}, age {age}.",      # test-only
]

def make_rows(n, split="train", seed=0):
    """Return n chat examples in TRL's prompt-completion format."""
    rng = random.Random(seed)
    test = split == "test"
    names = NAMES[20:] if test else NAMES[:20]
    cities = CITIES[16:] if test else CITIES[:16]
    templates = TEMPLATES[8:] if test else TEMPLATES[:8]
    rows = []
    for _ in range(n):
        person = {"name": rng.choice(names), "age": rng.randint(18, 80), "city": rng.choice(cities)}
        text = rng.choice(templates).format(**person)
        rows.append({
            "prompt": [{"role": "user", "content": f"{PROMPT}\n\nText: {text}"}],
            "completion": [{"role": "assistant", "content": json.dumps(person)}],
        })
    return rows

In [ ]:
def score(model, tokenizer, rows, show=3):
    """Greedy-generate a reply for each row; report valid-JSON rate and field accuracy."""
    model.eval()
    valid, fields_right = 0, 0
    for i, row in enumerate(rows):
        gold = json.loads(row["completion"][0]["content"])
        inputs = tokenizer.apply_chat_template(row["prompt"], add_generation_prompt=True,
                                               return_tensors="pt", return_dict=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=40, do_sample=False)
        reply = tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        try:
            pred = json.loads(reply)
            valid += 1
        except json.JSONDecodeError:
            pred = {}
        if isinstance(pred, dict):
            fields_right += sum(pred.get(k) == v for k, v in gold.items())
        if i < show:
            print(f"  reply: {reply!r}")
    n = len(rows)
    result = {"valid_json": valid / n, "field_accuracy": fields_right / (3 * n)}
    print(result)
    return result

## 1 · The model and two sets of rows
- `train_sample`: 40 rows the model **will train on**.
- `test_rows`: 40 **held-out** rows. Their names, cities and phrasings never appear in training.

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"
tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float32).to(device)

train_rows = make_rows(500, "train", seed=0)   # the model trains on all 500
train_sample = train_rows[:40]                 # 40 of them, to score
test_rows = make_rows(40, "test", seed=1)      # held-out
print(len(train_rows), "training rows |", len(train_sample), "scored training rows |", len(test_rows), "test rows")

## 2 · Score the base model first
Training changes `model` in place, so measure the base model **now**. These are the slowest calls in the notebook: the base model rambles until `max_new_tokens`.

**TODO 1:** Score the base model on the held-out rows (`base_test`) and on the training sample (`base_train`).

In [ ]:
# TODO 1
base_test = ...
base_train = ...

In [ ]:
# check 1
ok = all(isinstance(r, dict) and "valid_json" in r for r in (base_test, base_train))
check(ok and base_test != base_train, "Two baseline numbers, from two different sets of rows.",
      "Use the course score() function once per set of rows from section 1.")

## 3 · Fine-tune with Lesson 5's recipe
Nothing new here. It took about 20 seconds on my laptop GPU (RTX 3050 Ti).

In [ ]:
from trl import SFTConfig, SFTTrainer

args = SFTConfig(output_dir="sft-eval", num_train_epochs=1, per_device_train_batch_size=8,
                 learning_rate=5e-5, bf16=False, max_length=256,
                 logging_steps=10, save_strategy="no", report_to="none")
trainer = SFTTrainer(model=model, args=args, train_dataset=Dataset.from_list(train_rows),
                     processing_class=tok)
trainer.train()
tuned = trainer.model

## 4 · Score the fine-tuned model on the same rows
Same function, same rows, greedy decoding. Only the weights changed, so any difference is the fine-tune.

**TODO 2:** Score `tuned` on the held-out rows (`tuned_test`) and on the training sample (`tuned_train`).

In [ ]:
# TODO 2
tuned_test = ...
tuned_train = ...

In [ ]:
# check 2
ok = all(isinstance(r, dict) and "valid_json" in r for r in (tuned_test, tuned_train))
check(ok and tuned_test["valid_json"] > base_test["valid_json"],
      "Scored. Now read the table below.",
      "Score the fine-tuned model on the same two sets of rows you used in TODO 1.")

In [ ]:
print("| model      | train: valid JSON | train: fields | test: valid JSON | test: fields |")
print("|------------|-------------------|---------------|------------------|--------------|")
for name, tr, te in [("base", base_train, base_test), ("fine-tuned", tuned_train, tuned_test)]:
    print(f"| {name:<10} | {tr['valid_json']:>17.2f} | {tr['field_accuracy']:>13.2f} | "
          f"{te['valid_json']:>16.2f} | {te['field_accuracy']:>12.2f} |")

**Read the table.**
- Which column belongs in a model card? Why not the other one?
- The base model never trained on anything, yet its two columns differ. What does that tell you about comparing numbers measured on different rows?
- The test columns may say 1.00. Do you believe the model is perfect?

## 5 · Stress tests: go looking for failures
The held-out rows come from the same generator as the training rows. Real messages don't. A **stress test** is an input you write on purpose to break the pattern, together with the JSON you expect.

`run_stress` sends each message through the same prompt as training and compares every field.

In [ ]:
def run_stress(model, cases):
    """Greedy-generate a reply for each (text, expected) case and print PASS/FAIL."""
    results = []
    for i, (text, expected) in enumerate(cases):
        messages = [{"role": "user", "content": f"{PROMPT}\n\nText: {text}"}]
        inputs = tok.apply_chat_template(messages, add_generation_prompt=True,
                                         return_tensors="pt", return_dict=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=40, do_sample=False)
        reply = tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        try:
            pred, valid = json.loads(reply), True
        except json.JSONDecodeError:
            pred, valid = {}, False
        passed = isinstance(pred, dict) and all(pred.get(k) == v for k, v in expected.items())
        results.append({"text": text, "reply": reply, "valid_json": valid, "passed": passed})
        print(f"[{i}] {'PASS' if passed else 'FAIL'}  valid JSON: {valid}\n    text:  {text}\n    reply: {reply}")
    return results

**TODO 3:** Add **at least 5** cases of your own to `STRESS`, one short line each. Each case is `(message, expected JSON)`. Use `None` for a value the message doesn't give.

Ideas: a field left out, a second number in the sentence, a two-word city, a full name, typos, another language, a date of birth instead of an age. Two examples are already there.

In [ ]:
STRESS = [
    ("Hana is thirty-two and lives in Perth.", {"name": "Hana", "age": 32, "city": "Perth"}),  # age in words
    ("I'm Leo, 29.", {"name": "Leo", "age": 29, "city": None}),                               # no city given
    # TODO 3: add at least 5 cases of your own, one per line
    ...,
]
cases = [c for c in STRESS if c is not ...]
results = run_stress(tuned, cases)

In [ ]:
# check 3
mine = cases[2:]
well_formed = all(isinstance(c, tuple) and len(c) == 2 and isinstance(c[0], str)
                  and isinstance(c[1], dict) and set(c[1]) == {"name", "age", "city"} for c in mine)
check(len(mine) >= 5 and well_formed, f"{len(mine)} cases of your own.",
      "Add at least 5 (message, dict) pairs. Each dict needs exactly the keys name, age and city.")
if len(mine) >= 5:
    my_fails = sum(not r["passed"] for r in results[2:])
    check(my_fails > 0, f"{my_fails} of your cases broke the model. Now name them.",
          "All of your cases passed. Push harder: leave a field out, or add a number that isn't the age.")

## 6 · Name each failure
Give every FAIL one category. The category says what to fix:

| tag | what happened | the fix it points to |
|---|---|---|
| `"format"` | the reply isn't valid JSON | more or cleaner examples of the output format |
| `"invented"` | the message lacks the value; the model made one up | training rows with that field missing → `null` |
| `"wrong_value"` | the value is in the message; the model copied the wrong thing or only part of it | training rows with that pattern (two-word cities, extra numbers, ...) |
| `"out_of_scope"` | the schema has no right answer ("born in 1990", two people) | decide a rule; write it in the prompt and the model card |

**TODO 4:** Fill `TAGS` with `{case number: tag}` for every case that failed. The numbers are the `[i]` in the output above.

In [ ]:
CATEGORIES = {"format", "invented", "wrong_value", "out_of_scope"}

# TODO 4
TAGS = ...

In [ ]:
# check 4
failed = [i for i, r in enumerate(results) if not r["passed"]]
ok = isinstance(TAGS, dict) and all(TAGS.get(i) in CATEGORIES for i in failed)
check(ok, "Every failure has a name.",
      "TAGS needs one entry per FAIL, keyed by its [i] number, with a tag from CATEGORIES.")

In [ ]:
from collections import Counter

if isinstance(TAGS, dict):
    tally = Counter(TAGS[i] for i in failed if i in TAGS)
    print(f"{len(failed)} of {len(results)} stress tests failed. Valid JSON in {sum(r['valid_json'] for r in results)} of {len(results)} replies.")
    for tag, n in tally.most_common():
        print(f"  {tag:<13} {n}")

**Keep this tally.** The biggest category is the first thing to fix in your capstone (Lesson 10), where you design the training data yourself.

## Stretch (optional)
1. **Fix one category.** Write a few training rows where the city is missing and the completion says `"city": null`. Add them to `train_rows`, rerun sections 1–5, and see whether the `invented` failures go away.
2. **Does more training always help?** Change `500` to `64` in section 1 and rerun. Compare the table and the stress results.